# Synthetic chess RGB-D reconstruction on A100

Create `chess-colab-input.zip` with `python scripts/package_colab.py --out /path/to/chess-colab-input.zip`, then upload it here. This uses the bundled three-piece synthetic fixture. Add `HF_TOKEN` through Colab Secrets and select A100.

This template runs SAM 3, SAM 3D, geometry validation, simulation, scene graph export, and Scenic. It explicitly permits reported geometric fallback. No model weights or credentials are stored in this notebook. Board axes remain provisional until a physical a1 annotation is supplied. See `docs/STATUS.md` for actual validation limits.

Candidate frames must show the same stationary board and pieces. Missing or rejected objects are unknown, not verified empty squares. The pose fit uses only its reference; other views check depth and silhouette without changing mesh scale.


In [ ]:
import os, sys, json, subprocess, signal, time, zipfile
from pathlib import Path
import torch
from google.colab import userdata, files
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Select A100"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["MUJOCO_GL"] = "egl"
print({"gpu": torch.cuda.get_device_name(0), "credential_available": bool(os.environ["HF_TOKEN"])})


In [ ]:
uploaded = files.upload()  # Choose chess-colab-input.zip; do not upload a token file.
assert "chess-colab-input.zip" in uploaded
PACKAGE = Path("/content/chess-package")
PACKAGE.mkdir(exist_ok=False)
with zipfile.ZipFile("chess-colab-input.zip") as archive:
    for member in archive.infolist():
        if not (PACKAGE / member.filename).resolve().is_relative_to(PACKAGE.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(PACKAGE)
del uploaded
subprocess.run([sys.executable,"-m","pip","install","uv","huggingface_hub","-e",str(PACKAGE)],check=True)
ROOT = Path("/content/roboskills")
COMMIT = "eca4691d2dfbb808dd58f3dd279fed154de6828a"
if not ROOT.exists():
    subprocess.run(["git","clone","https://github.com/openrobokit/roboskills.git",str(ROOT)],check=True)
subprocess.run(["git","-C",str(ROOT),"checkout","--detach",COMMIT],check=True)
RUN = Path("/content") / ("chess-run-" + time.strftime("%Y%m%d-%H%M%S"))
RUN.mkdir()


In [ ]:
def execute(name, argv, timeout=1800):
    path = RUN / (name + ".log")
    started = time.monotonic()
    print(name, "starting", flush=True)
    with path.open("w") as log:
        p = subprocess.Popen(list(map(str,argv)), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            rc = p.wait(timeout=timeout)
        except BaseException:
            os.killpg(p.pid,signal.SIGKILL)
            p.wait()
            raise
    if rc:
        print(path.read_text(errors="replace")[-5000:].replace(os.environ["HF_TOKEN"],"[REDACTED]"))
        raise RuntimeError(name + " failed")
    print(name, "passed", round(time.monotonic()-started,1), "seconds", flush=True)
native = PACKAGE / "scripts/colab_sam3d.py"
execute("sam3d-setup",[sys.executable,native,"setup"])
execute("sam3d-weights",[sys.executable,native,"weights"])


In [ ]:
import numpy as np
capture = PACKAGE / "tests/fixtures/chess-small/capture.npz"
# These selected frames are asserted to show one stationary board arrangement.
with np.load(capture, allow_pickle=False) as recording:
    candidate_frames = list(range(len(recording["depth"])))
assert len(candidate_frames) >= 2, "Provide at least two distinct static views"
frames_file = RUN / "candidate_frames.json"
frames_file.write_text(json.dumps(candidate_frames))
workers = 2 if torch.cuda.get_device_properties(0).total_memory >= 70 * 2**30 else 1
execute("learned-chess", [sys.executable, "-m", "lidar_video.cli", "chess-reconstruct-gpu", capture,
    "--auto-calibrate",
    "--reference-frame", 0, "--candidate-frames", frames_file, "--static-scene",
    "--ambiguous-policy", "skip", "--workers", workers, "--native-script", native,
    "--mesh-policy", "fallback", "--out", RUN / "learned"], timeout=14400)


In [ ]:
from IPython.display import Video, display
result = RUN / "learned/result"
print((result / "verification.json").read_text())
display(Video(str(result / "simulation/rollout/mujoco_ext1.mp4"), embed=True))
display(Video(str(result / "sample_simulation/rollout/mujoco_ext1.mp4"), embed=True))


In [ ]:
import shutil
secret = os.environ["HF_TOKEN"]
for path in RUN.rglob("*.log"):
    path.write_text(path.read_text(errors="replace").replace(secret,"[REDACTED]"))
files.download(shutil.make_archive(str(RUN),"zip",RUN))
